# C. elegans: Load, Preprocess, and Window

This dataset-level notebook creates the immutable recording and temporal-window checkpoints shared by every method lane.
Confirm the raw-data path, array keys, sampling rate, behavior mapping, valid ranges, and gaps in the named dataset config before running.
Progress: long stages print checkpoint-aware timing; iterative stages also display live progress bars. Reused, failed, running, and missing checkpoints are reported explicitly.

Environment: select the project Python 3.14 `.venv` kernel. Install the locked packages with `uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks` from the repository root. The setup cell checks installed package versions; no upstream repository clone is required. Stored outputs may predate the backend migration; rerun the notebook to produce results with the current backend.


In [1]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    _REPOSITORY_ROOT = next(
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "notebooks" / "_shared.py").is_file()
    )
except StopIteration as error:
    raise RuntimeError(
        "Could not locate the effectome repository from the notebook working directory"
    ) from error
sys.path[:0] = [
    path
    for path in (str(_REPOSITORY_ROOT / "src"), str(_REPOSITORY_ROOT))
    if path not in sys.path
]


if sys.version_info[:2] != (3, 14):
    raise RuntimeError(
        "Select the project Python 3.14 .venv kernel before running this notebook. "
        f"Current interpreter: {sys.executable}"
    )
_BACKEND_DISTRIBUTIONS = ("fastcgc", "tigramite", "scikit-learn", "kmedoids", "igraph", "leidenalg")
try:
    _BACKEND_VERSIONS = {name: version(name) for name in _BACKEND_DISTRIBUTIONS}
except PackageNotFoundError as error:
    raise RuntimeError(
        "The notebook kernel is missing a required installed backend. Run "
        "uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks "
        "from the repository root, then restart the project .venv kernel."
    ) from error
print("Notebook interpreter:", sys.executable)
print("Installed backends:", _BACKEND_VERSIONS)


Notebook interpreter: /Users/sadiqadedayo/Documents/projects/effectome-to-behavior/effectome-to-behavior/.venv/bin/python3
Installed backends: {'fastcgc': '0.1.0', 'tigramite': '5.2.10.1', 'scikit-learn': '1.9.0', 'kmedoids': '0.5.5', 'igraph': '1.0.0', 'leidenalg': '0.12.0'}


In [2]:
from omegaconf import OmegaConf

from effectome.data_module import PreprocessConfig, WindowConfig
from notebooks._shared import (
    PROJECT_ROOT,
    make_run,
    print_notebook_header,
    print_stage_status,
    run_bundle_net_reference_preprocessing,
    run_preprocessing,
    stage_artifact_path,
)

DATASET_ID = "c_elegans"
DATASET_OUTPUT_ID = "bundle-net-c-elegans"
RECORDING_ID = "bundle-net-worm-0"
RUN_ID = "reference"
METHOD = "shared"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "analysis" / DATASET_OUTPUT_ID / RECORDING_ID
DATA_CONFIG_PATH = PROJECT_ROOT / "conf" / "data" / "c_elegans.yaml"
WINDOW_CONFIG_PATH = PROJECT_ROOT / "conf" / "windowing" / "primary_physical.yaml"
FORCE = False

DATA_CFG = OmegaConf.to_container(OmegaConf.load(DATA_CONFIG_PATH), resolve=True)
if not isinstance(DATA_CFG, dict):
    raise TypeError(f"Expected a mapping in {DATA_CONFIG_PATH}")
DATA_CFG["path"] = str((PROJECT_ROOT / str(DATA_CFG["path"])).resolve())
WINDOWING_CFG = OmegaConf.to_container(OmegaConf.load(WINDOW_CONFIG_PATH), resolve=True)
if not isinstance(WINDOWING_CFG, dict):
    raise TypeError(f"Expected a mapping in {WINDOW_CONFIG_PATH}")
if str(DATA_CFG.get("dataset_id", "")) != DATASET_OUTPUT_ID:
    raise ValueError(
        f"Notebook output dataset id {DATASET_OUTPUT_ID} does not match config dataset id "
        f"{DATA_CFG.get('dataset_id')}"
    )
if str(DATA_CFG.get("recording_id", "")) != RECORDING_ID:
    raise ValueError(
        f"Notebook output recording id {RECORDING_ID} does not match config recording id "
        f"{DATA_CFG.get('recording_id')}"
    )

PREPROCESS_CFG = PreprocessConfig(
    detrend=False,
    zscore=False,
    deconvolve=False,
    smooth_window=0,
    drop_low_variance=0.0,
)

BUNDLE_NET_REFERENCE_CFG = PreprocessConfig(
    **DATA_CFG["bundle_net_reference"]["preprocessing"]
)
BUNDLE_NET_TARGET_LENGTH = int(
    DATA_CFG["bundle_net_reference"]["paired_window_length"]
)
BUNDLE_NET_EXCLUDE_NEURONS = list(
    DATA_CFG["exclude_neuron_names"]
)
BUNDLE_NET_EXCLUDE_NAME_SOURCE = str(
    DATA_CFG["exclude_neuron_name_source"]
)

WINDOW_CFG = WindowConfig(**WINDOWING_CFG)
EXPECTED_BEHAVIOR_KEYS = ["motif"]

run = make_run(OUTPUT_ROOT, RUN_ID, METHOD)
print_notebook_header(
    run,
    title=f"{DATASET_ID} preprocessing",
    expected_stages=[
        "raw_recording",
        "recording",
        "windows",
        "bundle_net_reference_recording",
        "bundle_net_training_pairs"
    ],
    force=FORCE,
)


/Users/sadiqadedayo/Documents/projects/effectome-to-behavior/effectome-to-behavior/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


c_elegans preprocessing
run_id=reference method=shared
run_dir=/Users/sadiqadedayo/Documents/projects/effectome-to-behavior/effectome-to-behavior/outputs/analysis/bundle-net-c-elegans/bundle-net-worm-0/reference/shared
resume mode: FORCE=False, matching completed checkpoints will be reused
Rerun interrupted notebooks from the top; completed stages resume from checkpoints.
Stage status for reference/shared:
  - bundle_net_reference_recording: completed (signature=03acca0584, artifact=yes)
  - bundle_net_training_pairs: completed (signature=e3e31f6b76, artifact=yes)
  - raw_recording: completed (signature=8e0662f346, artifact=yes)
  - recording: completed (signature=979d31faee, artifact=yes)
  - windows: completed (signature=b76b88eae5, artifact=yes)


In [3]:
recording, windows = run_preprocessing(
    run,
    data_cfg=DATA_CFG,
    preprocess_cfg=PREPROCESS_CFG,
    window_cfg=WINDOW_CFG,
    force=FORCE,
)
missing_behavior = sorted(set(EXPECTED_BEHAVIOR_KEYS) - set(recording.behavior))
if missing_behavior:
    raise KeyError(f"Dataset is missing configured behavior keys: {missing_behavior}")

print(f"recording: {recording.n_neurons} neurons x {recording.n_timepoints} samples")
source_neuron_count = recording.metadata.get("source_neuron_count")
if source_neuron_count is not None:
    print(
        "neuron cohort: "
        f"{source_neuron_count} source -> {recording.n_neurons} analysis; "
        f"excluded={recording.metadata.get('excluded_neuron_names', [])}"
    )
print(f"windows: {windows.n_windows}; shape={windows.segments.shape}")
timing = windows.metadata["temporal_contract"]
print(
    "duration-first timing: "
    f"history={timing['history_length']} frames/{timing['history']['resolved_seconds']:.3f}s, "
    f"target={timing['target_length']} frames/{timing['target']['resolved_seconds']:.3f}s, "
    f"stride={timing['stride_length']} frames/{timing['stride']['resolved_seconds']:.3f}s, "
    f"adjacent-context overlap={1.0 - timing['stride_length'] / timing['history_length']:.1%}"
)
print(f"recording artifact: {stage_artifact_path(run, 'recording')}")
print(f"windows artifact: {stage_artifact_path(run, 'windows')}")
print_stage_status(run)


[reference/shared] raw_recording: checking checkpoint; computing if needed...
[reference/shared] raw_recording: completed in 0.4s
[reference/shared] recording: checking checkpoint; computing if needed...
[reference/shared] recording: completed in 0.0s
[reference/shared] windows: checking checkpoint; computing if needed...
[reference/shared] windows: completed in 0.2s
recording: 103 neurons x 3137 samples
neuron cohort: 109 source -> 103 analysis; excluded=['AVAR', 'AVAL', 'SMDVR', 'SMDVL', 'SMDDR', 'RIBL']
windows: 186; shape=(186, 349, 103)
duration-first timing: history=349 frames/120.153s, target=15 frames/5.164s, stride=15 frames/5.164s, adjacent-context overlap=95.7%
recording artifact: /Users/sadiqadedayo/Documents/projects/effectome-to-behavior/effectome-to-behavior/outputs/analysis/bundle-net-c-elegans/bundle-net-worm-0/reference/shared/stages/recording/artifact.pkl
windows artifact: /Users/sadiqadedayo/Documents/projects/effectome-to-behavior/effectome-to-behavior/outputs/anal

In [4]:
bundle_reference, bundle_pairs = run_bundle_net_reference_preprocessing(
    run,
    preprocess_cfg=BUNDLE_NET_REFERENCE_CFG,
    exclude_neuron_names=BUNDLE_NET_EXCLUDE_NEURONS,
    exclude_neuron_name_source=BUNDLE_NET_EXCLUDE_NAME_SOURCE,
    behavior_key="motif",
    target_length=BUNDLE_NET_TARGET_LENGTH,
    force=FORCE,
)
print(
    "Bundle-Net reference: "
    f"{bundle_reference.n_neurons} neurons x {bundle_reference.n_timepoints} samples; "
    f"paired windows={bundle_pairs.x_t.shape[0]}"
)
print(
    "reference recording artifact: "
    f"{stage_artifact_path(run, 'bundle_net_reference_recording')}"
)
print(f"training-pair artifact: {stage_artifact_path(run, 'bundle_net_training_pairs')}")
print_stage_status(run)


[reference/shared] bundle_net_reference_recording: checking checkpoint; computing if needed...
[reference/shared] bundle_net_reference_recording: completed in 0.0s
[reference/shared] bundle_net_training_pairs: checking checkpoint; computing if needed...
[reference/shared] bundle_net_training_pairs: completed in 0.1s
Bundle-Net reference: 103 neurons x 3137 samples; paired windows=3122
reference recording artifact: /Users/sadiqadedayo/Documents/projects/effectome-to-behavior/effectome-to-behavior/outputs/analysis/bundle-net-c-elegans/bundle-net-worm-0/reference/shared/stages/bundle_net_reference_recording/artifact.pkl
training-pair artifact: /Users/sadiqadedayo/Documents/projects/effectome-to-behavior/effectome-to-behavior/outputs/analysis/bundle-net-c-elegans/bundle-net-worm-0/reference/shared/stages/bundle_net_training_pairs/artifact.pkl
Stage status for reference/shared:
  - bundle_net_reference_recording: completed (signature=ea2e4ed460, artifact=yes)
  - bundle_net_training_pairs: 